# Mission 03: Fleet Demand Time Series Command

A taxi fleet operator must decide how many drivers to schedule each hour. You receive the **real** hourly demand of NYC yellow taxis for January and February 2024. Build the time-series view, find unusual days, and create a simple forecast that the operations team can trust.

**Lessons used:** 12 (time series), 05/14 (vectorised, efficient code), 17 (Polars lazy).

In [ ]:
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent.parent / "tools"))
import datasets

import polars as pl

def hourly_trips(month: str) -> pl.DataFrame:
    """Hourly trip counts of one month, with the cleaning rules of Lesson 08 (lazy Polars)."""
    start = pl.datetime(int(month[:4]), int(month[5:]), 1)
    dur = (pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime")).dt.total_seconds() / 60
    return (pl.scan_parquet(datasets.taxi(month))
              .filter(pl.col("tpep_pickup_datetime") >= start,
                      pl.col("tpep_pickup_datetime") < start.dt.offset_by("1mo"),
                      dur.is_between(1, 180), pl.col("fare_amount").is_between(3, 500))
              .group_by(pl.col("tpep_pickup_datetime").dt.truncate("1h").alias("hour"))
              .agg(trips=pl.len())
              .sort("hour")
              .collect())

hourly = pl.concat([hourly_trips("2024-01"), hourly_trips("2024-02")]).to_pandas().set_index("hour")["trips"]
print(len(hourly), "hours from", hourly.index.min(), "to", hourly.index.max())

## Tasks

1. **Complete the index.** Make sure every hour exists (fill missing hours with 0) and attach the `America/New_York` time zone.
2. **Daily view.** Resample to daily totals; add a 7-day rolling mean; plot both.
3. **Profiles.** Build an hour-of-week profile (168 values: Monday 00:00 … Sunday 23:00) from January. Plot it.
4. **Anomalies.** Flag days whose total deviates by more than 15 % from the median of the **same weekday** over the two months. List them and suggest a reason for each (check the calendar and the weather news of those days).
5. **Forecast.** Forecast every hour of 19–29 February with two methods: (a) the January hour-of-week profile, (b) "the same hour one week earlier" (seasonal naive). Compute the MAE and MAPE of both and plot one week.
6. **Recommendation.** Which method should operations use, and what would you improve next?

In [ ]:
# Tasks 1-6
...

## Submission Checklist

- [ ] The hourly index is complete and time-zone aware.
- [ ] Anomalies are computed with a rule you can explain.
- [ ] Both forecasts are evaluated on the same hours with MAE and MAPE.
- [ ] A clear recommendation.

A solution is in `03-Fleet-Demand-Time-Series-Command-solution.ipynb`. Try the mission first.